# Best-of-N Speed Benchmark Across Quantization Levels

Benchmark Best-of-N generation speed on the same set of MATH
problems across multiple quantization settings, such as fp16 and
GPTQ-int4 model variants.

Each configuration is loaded with vLLM, warmed up, timed for
num_trials runs, and then unloaded before the next configuration is
tested. This keeps the comparison focused on how quantization affects
throughput under the same benchmark settings.

Use this notebook to compare speed across precision or quantization
levels. Use benchmark_speed_bon_models_v1.ipynb for the separate
model-axis sweep at a fixed precision.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL + 1)

import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

import statistics

from sal.config import Config

from utils.load_data import load_data_hf
from unittests.notebook_utils import gpu_mem_used_gb, benchmark_bon_speed_llm_quant

In [2]:
# Dataset path
base_dir = '/groups/chichengz/tnn/datasets'

ds_split = "test"
ds_dir = os.path.join(base_dir, "prm800k/math_splits")

In [3]:
# Quantization configs to benchmark.
# GPTQ requires a pre-quantized model directory (point model_dir to it).
def _cfg(name, subdir, quantization, dtype):
    return {
        "name":         name,
        "model_dir":    os.path.join(base_dir, subdir),
        "quantization": quantization,
        "load_format":  "auto",
        "dtype":        dtype,
    }


quant_configs = [
    _cfg("llama-3b fp16",     "Llama3.2-3B-Instruct",          None,   "float16"),
    _cfg("llama-3b gptq",     "Llama3.2-3B-Instruct-GPTQ",     "gptq", "auto"),
    _cfg("qwen-3b fp16",      "Qwen2.5-3B-Instruct",           None,   "float16"),
    _cfg("qwen-3b gptq-int4", "Qwen2.5-3B-Instruct-GPTQ-Int4", "gptq", "auto"),
    _cfg("qwen-7b fp16",      "Qwen2.5-7B-Instruct",           None,   "float16"),
    _cfg("qwen-7b gptq-int4", "Qwen2.5-7B-Instruct-GPTQ-Int4", "gptq", "auto"),
]

In [4]:
# Best-of-N search params
config = Config()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048
config.n = 256
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

# Benchmark knobs
level = 4                          # MATH difficulty level
MAX_QUESTIONS = 5                  # cap on questions per benchmark
num_trials = 2                     # timed runs per config
warmup = 1                         # untimed warmup runs per config
llm_gpu_memory_utilization = 0.5

In [5]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)
num_questions = min(len(dataset), MAX_QUESTIONS)
batch_of_questions = [dataset[i]['problem'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

num_questions = 5


## Run benchmark

One config at a time; teardown between iterations frees the vLLM
engine before the next is loaded.

In [7]:
results = []
for qcfg in quant_configs:
    name, mem, times = benchmark_bon_speed_llm_quant(
        qcfg, config, batch_of_questions, num_trials,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        warmup=warmup,
    )
    results.append((name, mem, times))


=== llama-3b fp16 ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.64s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.59s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.75s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  GPU memory used: 16.53 GB
  trial 0:  303.21s total, 60.6417s/question
  trial 1:  291.90s total, 58.3801s/question


[rank0]:[W612 18:44:24.682559277 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== llama-3b gptq ===


...

[rank0]:[W612 21:02:49.468865517 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())



=== qwen-7b gptq-int4 ===


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  1.77it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  2.60it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  2.43it/s]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


  GPU memory used: 16.12 GB
  trial 0:  470.86s total, 94.1727s/question
  trial 1:  466.66s total, 93.3314s/question


[rank0]:[W612 21:27:07.219046895 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


## Summary

In [8]:
print(
    f"=== Summary (level={level}, "
    f"n_questions={num_questions}, n_trials={num_trials}) ==="
)
header = (
    f"{'quantization':<25}{'gpu (GB)':>10}"
    f"{'mean s/trial':>14}{'std':>8}{'s/question':>14}"
)
print(header)
print('-' * len(header))
for name, mem, times in results:
    mean = statistics.mean(times)
    std = statistics.stdev(times) if len(times) > 1 else 0.0
    print(
        f"{name:<25}{mem:>10.2f}"
        f"{mean:>14.2f}{std:>8.2f}{mean/num_questions:>14.4f}"
    )

=== Summary (level=4, n_questions=5, n_trials=2) ===
quantization               gpu (GB)  mean s/trial     std    s/question
-----------------------------------------------------------------------
llama-3b fp16                 16.53        297.55    8.00       59.5109
llama-3b gptq                 16.63        368.36    2.75       73.6720
qwen-3b fp16                  16.61        610.16   14.94      122.0322
qwen-3b gptq-int4             16.55        595.16    1.43      119.0318
qwen-7b fp16                  16.12       1184.41   43.85      236.8827
qwen-7b gptq-int4             16.12        468.76    2.97       93.7520
